## Narrative Position

**Pipeline step:** 회귀 추정과 instability (6/6) — **우리 팀 차별점 layer 3**

**This notebook answers:** *"verbosity·차원 통제 후에도 텍스트 feature가 KCGS 등급과 연관되는가? 모형 선택에 따라 결론이 어떻게 달라지는가?"*

**Next:** `06_alpha_analysis_v2.ipynb`

**Linked robustness evidence (★ 이 단계의 핵심 instability 증거):**
- `../jiwoo/05_sample_definition_sensitivity.ipynb` — N=30→N=210 N-curve, sign reversal zone
- `data/07_regression/robustness_by_exp.csv` — exp_B/E/F별 회귀 결과 비교
- `data/07_regression/vif_report.csv`, `cooks_distance.csv` — 다중공선성 · 영향관측치 진단

**Evaluation rubric coverage:** 모형 선택 · 해석과 한계

**Why this matters for our team's framing:** OLS/Ordered/Binary Logit 세 모형의 *계수 부호와 유의성이 일치하는가*가 신호 안정성의 핵심 지표다. 이 노트북은 모형 간 불일치를 *숨기지 않고 비교*한다. 또 N을 늘릴수록 부호가 뒤집히는 현상(Alpha 3 sign reversal)은 다음 노트북에서 cheap-talk 가설과 연결된다.

**금지 어휘 reminder:** "예측한다 / improves ESG / causes / true ESG"는 사용하지 않는다. "associated with / consistent with / weak-but-significant"로 일관한다.

---


# 05 · Regression v2 — OLS + Ordered Logit + Binary Logit + 진단

**Input**  : `data/05_features/merged_{exp_id}.parquet`  
**Output** : `data/07_regression/regression_results.parquet`  
           + `data/07_regression/vif_report.csv`  
           + `data/07_regression/bootstrap_coef_ci.parquet`  
           + `outputs/tables/regression_summary.html`

---

## 이 단계가 답하는 한 줄

> "ESG 어휘 비율이 분량·산업·연도를 통제한 후에도 KCGS 등급과 연관되는가?"

## 5개 모형 명세

| 모델 | 종속변수 | 독립변수 | 질문 |
|---|---|---|---|
| M1 (OLS) | `kcgs_grade_7` | `g_signal_ratio` only | naive correlation |
| M2 (OLS) | `kcgs_grade_7` | `+ log_tokens` | 분량 통제 후 |
| M3 (OLS) | `kcgs_grade_7` | `+ log_tokens + industry_FE + year_FE` | full controls |
| M4 (Ordered Logit) | `kcgs_grade_7` | M3와 동일 | 순서형 가정 robustness |
| M5 (Binary Logit) | `kcgs_grade_high` | M3와 동일 | A이상 vs 아래 |

**해석 원칙:**
- "causes" / "improves" / "predicts ESG" 표현 금지
- "associated with" / "consistent with" / "positively related" 사용
- M1 → M3로 계수가 어떻게 바뀌는지가 핵심 (verbosity bias 진단)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── matplotlib: Jupyter 환경에서는 use('Agg') 없이 직접 import
import matplotlib.pyplot as plt
import matplotlib
plt.rcParams['figure.dpi'] = 100

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

try:
    from pipeline_config import (
        D_FEATURES, D_REGRESS, O_TABLES, O_FIGURES,
        RANDOM_SEED, PRIMARY_EXP, ROBUSTNESS_EXP, N_BOOTSTRAP,
        GRADE_TO_7, GRADE_TO_4,
    )
    print("[OK] pipeline_config loaded")
except ImportError as e:
    print(f"[WARN] pipeline_config not found: {e}")
    # fallback
    D_FEATURES  = PROJECT_ROOT / 'data/05_features'
    D_REGRESS   = PROJECT_ROOT / 'data/07_regression'
    O_TABLES    = PROJECT_ROOT / 'outputs/tables'
    O_FIGURES   = PROJECT_ROOT / 'outputs/figures'
    RANDOM_SEED = 42
    PRIMARY_EXP = 'exp_F'
    ROBUSTNESS_EXP = ['exp_B', 'exp_E', 'exp_F']
    N_BOOTSTRAP = 1000
    GRADE_TO_7  = {'A+':7,'A':6,'B+':5,'B':4,'C':3,'D':2}
    GRADE_TO_4  = {'A+':4,'A':3,'B+':2,'B':1,'C':1,'D':1}

for d in [D_REGRESS, O_TABLES, O_FIGURES]:
    Path(d).mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
print(f"matplotlib version : {matplotlib.__version__}")
print(f"Regression output  : {D_REGRESS}")


## 1. 데이터 로드 + 전처리

In [ ]:
def load_regression_df(exp_id: str) -> pd.DataFrame:
    """
    회귀 분석용 DataFrame 준비.
    - KCGS NaN 제거 (0으로 채우지 않음)
    - industry_FE: induty_code 없으면 corp_code 앞 2자리로 근사
    - year_FE: fiscal_year dummy
    """
    # parquet 우선, csv fallback
    path_p = D_FEATURES / f"merged_{exp_id}.parquet"
    path_c = D_FEATURES.parent / "05_merged" / f"merged_{exp_id}.csv"
    
    if path_p.exists():
        df = pd.read_parquet(path_p)
    elif path_c.exists():
        df = pd.read_csv(path_c, dtype={"stock_code": str, "corp_code": str,
                                        "rcept_no": str})
        df["stock_code"] = df["stock_code"].astype(str).str.zfill(6)
    else:
        raise FileNotFoundError(f"merged_{exp_id} 없음")
    
    # KCGS 수치화
    if "kcgs_grade_7" not in df.columns and "kcgs_grade" in df.columns:
        df["kcgs_grade_7"]   = df["kcgs_grade"].map(GRADE_TO_7)
        df["kcgs_grade_4"]   = df["kcgs_grade"].map(GRADE_TO_4)
        df["kcgs_grade_high"] = df["kcgs_grade"].isin(["A","A+"]).astype(int)
    
    # log_tokens
    if "log_tokens" not in df.columns and "total_tokens" in df.columns:
        df["log_tokens"] = np.log(df["total_tokens"] + 1)
    
    # industry proxy: induty_code 없으면 corp_code 앞 2자리
    if "induty_code" not in df.columns:
        if "corp_code" in df.columns:
            df["industry_proxy"] = df["corp_code"].astype(str).str[:2]
        else:
            df["industry_proxy"] = "00"
    else:
        df["industry_proxy"] = df["induty_code"].astype(str).str[:2]
    
    # fiscal_year str (dummy용)
    df["year_str"] = df["fiscal_year"].astype(str)
    
    # 회귀 표본: KCGS 유효 행만
    df = df.dropna(subset=["kcgs_grade_7"]).copy()
    print(f"[{exp_id}] 회귀 표본 N={len(df)}")
    
    # g_signal_ratio 없으면 계산
    if "g_signal_ratio" not in df.columns and "joined_text" in df.columns:
        print("  g_signal_ratio 재계산...")
        from pipeline_config import G_SIGNAL_SET
        def _g_ratio(txt):
            toks = str(txt).split()
            if not toks: return None
            return sum(1 for t in toks if t in G_SIGNAL_SET) / len(toks)
        df["g_signal_ratio"] = df["joined_text"].apply(_g_ratio)
    
    return df


df_reg = load_regression_df(PRIMARY_EXP)
print(f"\n컬럼: {df_reg.columns.tolist()}")
print(f"\n등급 분포:")
if "kcgs_grade" in df_reg.columns:
    print(df_reg["kcgs_grade"].value_counts().sort_index())

## 2. VIF 진단 — 다중공선성 사전 확인

In [ ]:
def compute_vif(df: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    """
    VIF > 10 → 다중공선성 위험 (해당 변수 해석 주의)
    VIF > 5  → 경계 수준
    """
    valid = df[feature_cols].dropna()
    X = sm.add_constant(valid)
    
    vif_data = []
    for i, col in enumerate(X.columns):
        if col == "const":
            continue
        vif = variance_inflation_factor(X.values, i)
        vif_data.append({"variable": col, "VIF": round(vif, 3)})
    
    vif_df = pd.DataFrame(vif_data).sort_values("VIF", ascending=False)
    vif_df["flag"] = vif_df["VIF"].apply(
        lambda v: "HIGH" if v > 10 else ("MODERATE" if v > 5 else "OK")
    )
    return vif_df


# g_signal_ratio + log_tokens 의 VIF
vif_cols = [c for c in ["g_signal_ratio","esg_signal_ratio",
                         "esg_tfidf_concentration","log_tokens"]
            if c in df_reg.columns]

vif_df = compute_vif(df_reg, vif_cols)
print("=== VIF Report ===")
display(vif_df)

vif_path = D_REGRESS / "vif_report.csv"
vif_df.to_csv(vif_path, index=False, encoding="utf-8-sig")
print(f"→ saved: {vif_path}")

# 핵심 경고
high_vif = vif_df[vif_df["flag"]=="HIGH"]
if len(high_vif) > 0:
    print(f"\n⚠️ HIGH VIF 변수: {high_vif['variable'].tolist()} — 회귀 계수 해석 주의")
else:
    print("\n✓ VIF < 10 — 다중공선성 허용 범위")

## 3. OLS 회귀 (M1 ~ M3)

In [ ]:
# industry dummy (기준: 가장 빈도 높은 산업)
df_reg["industry_FE"] = pd.Categorical(df_reg["industry_proxy"])
df_reg["year_FE"]     = pd.Categorical(df_reg["year_str"])

def run_ols(df, formula, cov_type="HC1", label=""):
    """
    OLS 회귀 실행.
    cov_type='HC1': Heteroskedasticity-robust SE (White's correction)
    왜 robust SE: KCGS 등급을 연속변수로 취급 + 패널 데이터 → 오차항 비등분산 가능성
    """
    model = smf.ols(formula, data=df).fit(cov_type=cov_type)
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  N={model.nobs:.0f}  R²={model.rsquared:.4f}  Adj-R²={model.rsquared_adj:.4f}")
    print(f"  F-stat={model.fvalue:.3f}  p={model.f_pvalue:.4f}")
    print(f"{'='*60}")
    
    # 핵심 계수만 출력
    coef_df = pd.DataFrame({
        "coef":   model.params,
        "se":     model.bse,
        "t":      model.tvalues,
        "p":      model.pvalues,
        "ci_lo": model.conf_int()[0],
        "ci_hi": model.conf_int()[1],
    }).round(4)
    
    # industry/year FE 행 제외하고 출력
    key_rows = coef_df[~coef_df.index.str.contains("T\.", na=False)]
    print(key_rows.to_string())
    
    return model, coef_df

# M1: naive
m1, m1_coef = run_ols(
    df_reg, "kcgs_grade_7 ~ g_signal_ratio",
    label="M1 (OLS) — g_signal_ratio only"
)

# M2: + log_tokens
m2, m2_coef = run_ols(
    df_reg, "kcgs_grade_7 ~ g_signal_ratio + log_tokens",
    label="M2 (OLS) — + log_tokens"
)

# M3: full controls
# industry FE는 C() notation으로
if df_reg["industry_proxy"].nunique() > 1:
    formula_m3 = "kcgs_grade_7 ~ g_signal_ratio + log_tokens + C(industry_proxy) + C(year_str)"
else:
    formula_m3 = "kcgs_grade_7 ~ g_signal_ratio + log_tokens + C(year_str)"

m3, m3_coef = run_ols(
    df_reg, formula_m3,
    label="M3 (OLS) — full controls (log_tokens + industry_FE + year_FE)"
)

## 4. Ordered Logit (M4) — 순서형 종속변수 robustness

In [ ]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

def run_ordered_logit(df: pd.DataFrame, feature_cols: list, target_col: str = "kcgs_grade_7"):
    """
    Ordered Logit / Proportional Odds Model.
    
    왜 Ordered Logit인가:
        KCGS grade는 순서형이다. OLS는 등간척도를 가정하므로
        Ordered Logit이 이론적으로 더 적합.
        단, 계수 해석이 Log-Odds Ratio로 바뀐다.
    """
    valid_cols = [c for c in feature_cols if c in df.columns]
    df_valid = df[[target_col] + valid_cols].dropna()
    
    # target을 정수로
    y = df_valid[target_col].astype(int)
    X = df_valid[valid_cols]
    X = (X - X.mean()) / X.std()   # standardize for convergence
    
    try:
        mod = OrderedModel(y, X, distr='logit')
        result = mod.fit(method='bfgs', disp=False)
        
        print("\n" + "="*60)
        print("  M4 (Ordered Logit) — standardized features")
        print("="*60)
        print(f"  N={len(y)}  Log-Likelihood={result.llf:.2f}  AIC={result.aic:.2f}")
        print("\n  Coefficients (Log-Odds):")
        
        coef_df = pd.DataFrame({
            "coef":   result.params,
            "se":     result.bse,
            "z":      result.tvalues,
            "p":      result.pvalues,
        }).round(4)
        
        # threshold 파라미터 분리
        feat_coefs = coef_df[coef_df.index.isin(valid_cols)]
        print(feat_coefs.to_string())
        
        return result, coef_df
    except Exception as e:
        print(f"Ordered Logit 수렴 실패: {e}")
        return None, None

ol_feat_cols = [c for c in ["g_signal_ratio","log_tokens","esg_signal_ratio",
                              "esg_tfidf_concentration"]
                if c in df_reg.columns]

m4_result, m4_coef = run_ordered_logit(df_reg, ol_feat_cols)

## 5. Binary Logit (M5) — A이상 vs 아래

In [ ]:
def run_binary_logit(df: pd.DataFrame, feature_cols: list, target_col: str = "kcgs_grade_high"):
    """
    Logistic Regression: target=1 (A or A+), target=0 (B+ 이하).
    
    왜 Binary인가:
        'A 이상 등급을 받는 firm'은 실질적으로 의미 있는 경계.
        연속형 OLS나 순서형 Logit과 결과 방향이 같으면 robust.
    """
    if target_col not in df.columns:
        df[target_col] = df["kcgs_grade_7"].apply(lambda x: 1 if x >= 6 else 0)
    
    valid_cols = [c for c in feature_cols if c in df.columns]
    df_valid = df[[target_col] + valid_cols].dropna()
    
    print(f"\nBinary target dist: {df_valid[target_col].value_counts().to_dict()}")
    
    formula = f"{target_col} ~ {' + '.join(valid_cols)}"
    try:
        model = smf.logit(formula, data=df_valid).fit(disp=False)
        
        print("\n" + "="*60)
        print("  M5 (Binary Logit) — kcgs_grade_high (A/A+=1)")
        print("="*60)
        print(f"  N={model.nobs:.0f}  Pseudo-R²={model.prsquared:.4f}")
        print(f"  Log-Likelihood={model.llf:.2f}  AIC={model.aic:.2f}")
        
        coef_df = pd.DataFrame({
            "coef":    model.params,
            "odds_ratio": np.exp(model.params),
            "se":      model.bse,
            "z":       model.tvalues,
            "p":       model.pvalues,
        }).round(4)
        print(coef_df.to_string())
        
        return model, coef_df
    except Exception as e:
        print(f"Binary Logit 오류: {e}")
        return None, None

m5_result, m5_coef = run_binary_logit(df_reg, ["g_signal_ratio","log_tokens"])

## 6. Bootstrap CI for OLS Coefficients (M2 기준)

In [ ]:
def bootstrap_ols_coef(
    df: pd.DataFrame,
    formula: str,
    n_boot: int = N_BOOTSTRAP,
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """Bootstrap resample → OLS 계수 분포 → 95% CI"""
    rng = np.random.default_rng(seed)
    n = len(df)
    boot_coefs = []
    
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        sample = df.iloc[idx]
        try:
            res = smf.ols(formula, data=sample).fit()
            boot_coefs.append(res.params.to_dict())
        except:
            pass
    
    boot_df = pd.DataFrame(boot_coefs)
    ci_df = pd.DataFrame({
        "coef_obs":  smf.ols(formula, data=df).fit().params,
        "boot_mean": boot_df.mean(),
        "boot_std":  boot_df.std(),
        "ci_lo_95":  boot_df.quantile(0.025),
        "ci_hi_95":  boot_df.quantile(0.975),
    }).round(5)
    
    ci_df["ci_excludes_zero"] = (
        (ci_df["ci_lo_95"] > 0) | (ci_df["ci_hi_95"] < 0)
    ).astype(int)
    
    return ci_df


print(f"Bootstrap CI for M2 계수 (B={N_BOOTSTRAP})...")
boot_coef_df = bootstrap_ols_coef(
    df_reg.dropna(subset=["g_signal_ratio","log_tokens","kcgs_grade_7"]),
    formula="kcgs_grade_7 ~ g_signal_ratio + log_tokens",
)
print("\n=== Bootstrap CI for OLS coefficients (M2) ===")
display(boot_coef_df)

boot_coef_path = D_REGRESS / "bootstrap_coef_ci.parquet"
boot_coef_df.to_parquet(boot_coef_path)
print(f"→ saved: {boot_coef_path}")

## 7. 진단 — Heteroskedasticity + Influence

In [ ]:
# Breusch-Pagan test (이분산성)
print("=== Breusch-Pagan Test (M2) ===")
try:
    bp_data = df_reg[["g_signal_ratio","log_tokens","kcgs_grade_7"]].dropna()
    m2_bp = smf.ols("kcgs_grade_7 ~ g_signal_ratio + log_tokens", data=bp_data).fit()
    bp_stat, bp_p, f_stat, f_p = het_breuschpagan(
        m2_bp.resid,
        sm.add_constant(bp_data[["g_signal_ratio","log_tokens"]]).values
    )
    print(f"  LM stat = {bp_stat:.4f}  p = {bp_p:.4f}")
    if bp_p < 0.05:
        print("  → 이분산성 존재 — robust SE (HC1) 사용 정당화")
    else:
        print("  → 이분산성 기각 (그래도 robust SE 사용 권장)")
except Exception as e:
    print(f"  BP test 오류: {e}")

# Cook's Distance — 영향력 큰 관측치
print("\n=== Top-10 Influential Observations (Cook's D) ===")
try:
    influence = m2.get_influence()
    cooks_d = influence.cooks_distance[0]
    
    influence_df = bp_data.copy()
    influence_df["cooks_d"] = cooks_d
    
    if "stock_code" in df_reg.columns:
        influence_df["stock_code"] = df_reg.loc[influence_df.index, "stock_code"]
    if "fiscal_year" in df_reg.columns:
        influence_df["fiscal_year"] = df_reg.loc[influence_df.index, "fiscal_year"]
    
    top_influence = influence_df.nlargest(10, "cooks_d")[
        [c for c in ["stock_code","fiscal_year","g_signal_ratio",
                     "kcgs_grade_7","cooks_d"] if c in influence_df.columns]
    ]
    display(top_influence.round(4))
    
    threshold = 4 / len(influence_df)
    n_outlier = (cooks_d > threshold).sum()
    print(f"\nCook's D > 4/N ({threshold:.4f}): {n_outlier}개 관측치")
    
    cook_path = D_REGRESS / "cooks_distance.csv"
    influence_df.round(5).to_csv(cook_path, index=False, encoding="utf-8-sig")
    print(f"→ saved: {cook_path}")
except Exception as e:
    print(f"Cook's D 계산 오류: {e}")

## 8. M1~M5 계수 비교표 (보고서 Table 4)

In [ ]:
def extract_coef_row(model, label, key_var="g_signal_ratio"):
    if model is None:
        return {"model": label, "coef": None, "se": None, "p": None, "n": None, "r2": None}
    
    try:
        coef = model.params.get(key_var, None)
        se   = model.bse.get(key_var, None)
        p    = model.pvalues.get(key_var, None)
    except:
        coef, se, p = None, None, None
    
    r2 = getattr(model, "rsquared", getattr(model, "prsquared", None))
    
    return {
        "model": label,
        "coef_g_signal_ratio": round(coef, 4) if coef is not None else None,
        "se":   round(se,   4) if se   is not None else None,
        "p":    round(p,    4) if p    is not None else None,
        "n":    int(model.nobs) if hasattr(model, "nobs") else None,
        "r2":   round(r2, 4)  if r2 is not None else None,
        "controls": "",
    }

comparison_table = pd.DataFrame([
    {**extract_coef_row(m1, "M1 OLS"), "controls": "none"},
    {**extract_coef_row(m2, "M2 OLS"), "controls": "log_tokens"},
    {**extract_coef_row(m3, "M3 OLS"), "controls": "log_tokens + industry_FE + year_FE"},
    {**extract_coef_row(m4_result, "M4 Ordered Logit"), "controls": "log_tokens"},
    {**extract_coef_row(m5_result, "M5 Binary Logit"), "controls": "log_tokens"},
])

print("=== Regression Comparison Table ===")
display(comparison_table)

table_path = D_REGRESS / "regression_comparison.csv"
comparison_table.to_csv(table_path, index=False, encoding="utf-8-sig")
print(f"→ saved: {table_path}")

## 9. 전처리 Robustness — exp_B / exp_E / exp_F 동일 회귀

In [ ]:
robustness_rows = []
for exp_id in ROBUSTNESS_EXP:
    try:
        df_exp = load_regression_df(exp_id)
        df_exp_valid = df_exp.dropna(subset=["g_signal_ratio","log_tokens","kcgs_grade_7"])
        if len(df_exp_valid) < 10:
            continue
        
        res = smf.ols(
            "kcgs_grade_7 ~ g_signal_ratio + log_tokens",
            data=df_exp_valid
        ).fit(cov_type="HC1")
        
        robustness_rows.append({
            "exp_id":             exp_id,
            "n":                  int(res.nobs),
            "coef_g_signal":      round(res.params.get("g_signal_ratio", None), 4),
            "p_g_signal":         round(res.pvalues.get("g_signal_ratio", None), 4),
            "coef_log_tokens":    round(res.params.get("log_tokens", None), 4),
            "p_log_tokens":       round(res.pvalues.get("log_tokens", None), 4),
            "r2":                 round(res.rsquared, 4),
        })
    except Exception as e:
        print(f"[{exp_id}] 오류: {e}")

rob_df = pd.DataFrame(robustness_rows)
print("=== Preprocessing Robustness (same formula across exp_B/E/F) ===")
display(rob_df)

rob_path = D_REGRESS / "robustness_by_exp.csv"
rob_df.to_csv(rob_path, index=False, encoding="utf-8-sig")
print(f"→ saved: {rob_path}")

## 10. 계수 변화 시각화 (M1 → M2 → M3)

g_signal_ratio 계수가 log_tokens 통제 전후 어떻게 바뀌는가?

In [ ]:
models_viz = [
    ("M1\n(naive)",       m1),
    ("M2\n(+log_tokens)", m2),
    ("M3\n(full ctrl)",   m3),
]

coef_vals, coef_ci_lo, coef_ci_hi = [], [], []
labels = []

for lbl, mod in models_viz:
    if mod is None: continue
    try:
        c = mod.params["g_signal_ratio"]
        ci = mod.conf_int().loc["g_signal_ratio"]
        coef_vals.append(c)
        coef_ci_lo.append(c - ci[0])
        coef_ci_hi.append(ci[1] - c)
        labels.append(lbl)
    except:
        pass

if coef_vals:
    fig, ax = plt.subplots(figsize=(8, 4))
    x = range(len(labels))
    colors = ['crimson' if v < 0 else 'steelblue' for v in coef_vals]
    
    ax.bar(x, coef_vals, color=colors, alpha=0.75, width=0.4)
    ax.errorbar(x, coef_vals,
                yerr=[coef_ci_lo, coef_ci_hi],
                fmt='none', color='black', capsize=5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel("OLS coefficient of g_signal_ratio")
    ax.set_title("g_signal_ratio coefficient across model specifications\n(cheap-talk: negative = consistent)")
    
    plt.tight_layout()
    fig_path = str(O_FIGURES / "coefficient_comparison_M1_M3.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"→ figure saved: {fig_path}")

## 11. Interpretation — 핵심 읽는 법

**M1 → M2 계수 변화:**
- g_signal_ratio가 log_tokens 통제 후 *더 음수*가 됨 → verbosity bias가 +방향 편향을 주었음
- g_signal_ratio가 유의성이 소실됨 → g_signal_ratio가 주로 verbosity의 함수였음

**M3 계수 해석:**
- 부호 방향이 cheap-talk 가설과 정합인가?
- 유의성보다 CI 범위 전체를 보라. 'zero 포함 여부'

**다음 단계:** `06_alpha_analysis_v2.ipynb`
- Verbosity-Adjusted ESG Score (M2 잔차)
- Low-grade firm strategic disclosure (MWU by grade group)

## Robustness Inset — Sample Definition Sensitivity (N-curve)

회귀 결과의 stability는 "한 번의 추정"이 아니라 **표본 크기·구성을 바꿔도 같은 결론이 유지되는가**로 평가되어야 한다. 본 inset은 두 가지를 한 화면에 모은다:

1. **N-curve sign reversal** — `outputs/figures/alpha3_sign_reversal_exp_F.png` (Alpha 3 사전 figure)
2. **Preprocessing별 회귀 robustness** — `data/07_regression/robustness_by_exp.csv`

N-curve 원본 분석은 `../jiwoo/05_sample_definition_sensitivity.ipynb`에 있다 — 본 inset은 *요약 view*다.


In [ ]:
# === Robustness Inset · 05_regression_v2 ===
# 표본 크기와 preprocessing에 따른 회귀 결과 안정성을 한 화면에 모은다.
import pandas as pd
from pathlib import Path
from IPython.display import display, Image, Markdown

PROJ = Path.cwd()
while PROJ.name and not (PROJ/'data').exists() and PROJ.parent != PROJ:
    PROJ = PROJ.parent

# 1) N-curve sign reversal figure (Alpha 3에서 생성되어 있음)
sign_fig = PROJ/'outputs'/'figures'/'alpha3_sign_reversal_exp_F.png'
if sign_fig.exists():
    display(Markdown('**N-curve · Sign reversal zone (Alpha 3 figure):**'))
    display(Image(str(sign_fig)))
else:
    display(Markdown(f'⚠ `{sign_fig}` 없음. jiwoo/05_sample_definition_sensitivity 실행 필요.'))

# 2) preprocessing별 회귀 robustness 표
rob_path = PROJ/'data'/'07_regression'/'robustness_by_exp.csv'
if rob_path.exists():
    rob = pd.read_csv(rob_path)
    display(Markdown('**Robustness by preprocessing experiment:**'))
    display(rob)
else:
    display(Markdown(f'⚠ `{rob_path}` 없음.'))


### Interpretation — 우리 팀 framing

- 소표본(N ≈ 30)에서 양의 ρ로 보였던 신호가 N=210으로 확장하면 음으로 뒤집힌다. 이는 "분석 실패"가 아니라 *cheap-talk 가설의 정합성*을 보여주는 결과다 — 소표본에서는 KOSPI top-tier(고등급) firm이 우연히 더 많이 포함되어 양의 상관이 *지표 인공물*로 나타났고, 표본이 넓어지면서 **"낮은 등급일수록 ESG 어휘를 더 쓴다"**는 strategic disclosure 신호가 드러난다.
- preprocessing 실험을 가로질러 회귀 계수의 *방향성*이 흔들리는 feature는 **measurement fragility가 큰 후보**다. 이 feature를 보고서에서 강하게 주장하지 않는다.

**금지 어휘 reminder:** sign reversal을 "오류" "이상치" "잡음"으로 부르지 않는다. 우리는 이를 *empirical finding*으로 부른다.

## Decision Box — 회귀 모형 선택

- **Alternative:** OLS (M1~M3), Ordered Logit (M4), Binary Logit (M5)
- **Choice:** 세 모형 모두 보고. 어느 하나의 결과만 보고하지 않는다.
- **Justification:** ESG 등급은 ordinal이므로 Ordered Logit이 통계적으로 더 적절하지만, OLS는 계수의 직관적 해석을 제공하고, Binary Logit은 A 이상 여부의 임상적 가독성을 제공한다. 세 모형이 *동일한 방향성*을 보일 때만 신호로 인정한다.
- **Limitation:** OLS는 등급 간 간격 동일성 가정. Ordered Logit은 비례오즈 가정. Binary Logit은 등급 정보 손실. 세 모형 간 결론 차이가 있으면 그 차이 자체가 보고 대상이다.
